# Puckworks — Guided Espresso Pull (exploratory)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trbrewer/puckworks/blob/main/notebooks/guided_espresso_pull_colab.ipynb)

Enter a bounded recipe and run one **coherent, model-backed** espresso pull on a normal Colab **CPU**, then watch it stage by stage. This is a guided **mechanism explorer**, not an optimizer, a flavor predictor, or a digital twin.

**What this does — and does not — do:**

- One coherent **Cameron** extraction model is executed (grind → machine/flow → extraction → cup).
- `bean_label` is **descriptive only** — it never changes a number. `coffee_profile` is the model-backed parameter selector (only a documented `reference` profile exists).
- Pressure is a **prescribed constant input**, not a predicted dynamic profile.
- Temperature is **recorded-only** — the v0.3.0 primary model has no thermal transient.
- Puck **wetting and physical first drip are NOT modeled** (the model begins from a saturated bed).
- Alternative components are **not blended** into the primary chain; none are executed as lenses.
- Domain **warnings** flag evidence-range departures. This is **not** a taste predictor or a recipe optimizer, and it reports chemical **composition**, never sensory flavor.

## 1. Install the exact v0.3.0 public release (with hash verification)

By default this **downloads** the exact `puckworks-0.3.0-py3-none-any.whl` from the v0.3.0 GitHub Release, **verifies its SHA-256 against the recorded canonical hash**, and installs the verified local file — on a mismatch it refuses to install. It never installs an unreleased or mutable build and never falls back to the source checkout. Automated tests set `PUCKWORKS_WHEEL` to a locally built candidate wheel for a hermetic, network-free run. puckworks is **not on PyPI**. Figures use matplotlib (preinstalled on Colab; part of the `puckworks[viz]` extra).

In [ ]:
import hashlib
import os
import subprocess
import sys
import urllib.request

# Exact, immutable v0.3.0 GitHub Release asset. The download path is verified against this
# SHA-256 before installing; PUCKWORKS_WHEEL overrides it for hermetic (offline) testing.
RELEASED = True
WHEEL_FILENAME = 'puckworks-0.3.0-py3-none-any.whl'
WHEEL_URL = 'https://github.com/trbrewer/puckworks/releases/download/v0.3.0/puckworks-0.3.0-py3-none-any.whl'
WHEEL_SHA256 = 'a1da397b439dd0ee7bd008fdb3d3eebd83b2d5d0010f450e66e3a004ba4feacc'


def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


override = os.environ.get('PUCKWORKS_WHEEL', '').strip()
if override:
    # Hermetic / CI mode: install a locally built candidate wheel, no network, no hash gate
    # (a development wheel has a different hash than the eventual release).
    print('Hermetic mode — installing local candidate wheel:', override)
    if not os.path.isfile(override):
        raise SystemExit('PUCKWORKS_WHEEL is set but is not a file: ' + override)
    target = override
elif not RELEASED:
    raise SystemExit(
        'Development preview: the Guided Espresso Pull ships in puckworks v0.3.0, which is not '
        'released yet. This notebook does not install an unreleased or mutable build. Set '
        'PUCKWORKS_WHEEL to a locally built candidate wheel to run it now; public one-click '
        'execution is enabled when the v0.3.0 GitHub Release is published.')
else:
    print('Expected wheel SHA-256:', WHEEL_SHA256)
    target = os.path.join('/tmp', WHEEL_FILENAME)
    urllib.request.urlretrieve(WHEEL_URL, target)
    actual = _sha256(target)
    print('Verified wheel SHA-256:', actual)
    if actual != WHEEL_SHA256:
        raise SystemExit('Wheel SHA-256 mismatch — refusing to install (expected '
                         + WHEEL_SHA256 + ', got ' + actual + ').')
    print('Hash OK — installing the verified local file.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', target], check=True)
print('Install complete.')

## 2. Import and environment check

Confirm the installed package imported, and see exactly which build and interpreter you are on.

In [ ]:
import platform

import puckworks
import puckworks.product as pw

print('puckworks version :', puckworks.__version__)
print('Python version    :', platform.python_version())
print('Imported from     :', puckworks.__file__)

## 3. Choose your recipe

These are the controls the guided model can represent **honestly**. Values outside the documented evidence range are not blocked in *Warn* mode — the run is flagged as an extrapolation.

In [ ]:
#@title Recipe controls { run: "auto" }
dose_g = 20.0            #@param {type:"number"}
target_beverage_g = 40.0 #@param {type:"number"}
pressure_bar = 9.0       #@param {type:"number"}
grind_setting = 1.7      #@param {type:"number"}
coffee_profile = "reference"  #@param ["reference"]
bean_label = "my house espresso"  #@param {type:"string"}
domain_policy = "Warn and continue"  #@param ["Warn and continue", "Strict"]

# bean_label is descriptive metadata only — it never changes a numeric result.
# coffee_profile is the model-backed parameter selector (only 'reference' is documented).
_policy = 'strict' if domain_policy == 'Strict' else 'warn'

### Recorded context (does **not** affect the model)

Brew temperature is **recorded with the recipe but does not affect the current v0.3.0 primary model** (there is no thermal transient). Changing it is not a sensitivity; it is recorded context.

In [ ]:
#@title Recorded-only context { run: "auto" }
brew_temperature_c = 93.0  #@param {type:"number"}
# Recorded-only: temperature is echoed back with the recipe but is not a model input in v0.3.0.

In [ ]:
from dataclasses import replace

recipe, config = pw.load_pull_preset('guided_v1')
recipe = replace(recipe, dose_g=dose_g, target_beverage_g=target_beverage_g,
                 pressure_bar=pressure_bar, grind_setting=grind_setting,
                 brew_temperature_c=brew_temperature_c, coffee_profile=coffee_profile,
                 bean_label=bean_label)
config = replace(config, domain_policy=_policy)
print('Recipe :', recipe.dose_g, 'g in ->', recipe.target_beverage_g, 'g out ·',
      recipe.pressure_bar, 'bar ·', recipe.grinder_model, 'dial', recipe.grind_setting,
      '· profile', recipe.coffee_profile)
print('Bean   :', recipe.bean_label, '(label only)')
print('Temp   :', recipe.brew_temperature_c, 'C (recorded-only)')
print('Policy :', config.domain_policy)

## 4. Domain report (before the run)

The guided pull checks your recipe against hard physical/solver limits (**rejected**) and the documented evidence range (**warning**), and marks recorded-only inputs as **not applicable**. It never silently clamps a value.

In [ ]:
for f in pw.evaluate_domain(recipe):
    print(f'[{f.status.value:14}] {f.field:20} = {f.supplied_value}  ({f.plain_explanation})')

## 5. Run the guided pull

The simulation is sub-second, so the stage events are collected and printed after completion rather than animated.

In [ ]:
events = []
def _progress(event, payload):
    events.append((event.value, dict(payload)))

run = pw.simulate_pull(recipe, config, progress=_progress)
print('state:', run.completion_state, '· run id:', run.run_id)
print('\nStage events:')
for name, payload in events:
    print('  ', name, payload)

## 6. Stage-by-stage cards

Recipe → Grind → Bed → Machine → Wetting → Flow → Extraction → Cup. Executed stages show their method, inputs/outputs with units, evidence, assumptions, and caveats; unsupported stages are shown explicitly rather than omitted or simulated.

In [ ]:
for s in run.stages:
    print(f'=== {s.sequence}. {s.stage_id} — {s.method_name} [{s.evidence_badge}] ===')
    print('  method  :', s.method_plain)
    print('  inputs  :', {k: f"{v['value']} {v['unit']}" for k, v in s.inputs.items()})
    print('  outputs :', {k: f"{v['value']} {v['unit']}" for k, v in s.outputs.items()})
    print('  evidence:', s.evidence_strength, '· valid range:', s.valid_range)
    print('  caveat  :', s.caveat)
    print()
print('Wetting / infiltration: NOT MODELED in guided_v1 (the primary model begins saturated).')

## 7. Figures

Every figure carries an `EXPLORATORY_SIMULATION` badge, direct labels with units, and grayscale-readable line styles. Prescribed, simulated, and derived quantities are labelled distinctly. A static text equivalent is written to `guided_pull_captions.txt` and the Markdown report.

In [ ]:
from IPython.display import Image, display

artifacts = pw.render_pull_report(run, 'guided_pull_report', overwrite=True)
for png in (artifacts.summary_png, artifacts.pressure_flow_png,
            artifacts.cup_progress_png, artifacts.extraction_progress_png):
    display(Image(filename=png))

## 8. Results, coverage, and limits

In [ ]:
obs = run.final_observables
print('Supported observables:')
for k in ('extraction_yield_pct', 'tds_pct', 'shot_duration_s', 'beverage_mass_g',
          'extracted_mass_g', 'mean_flow_g_s', 'pressure_bar'):
    v = obs[k]
    print(f'  {k:24}: {v["value"]} {v["unit"]}')

print('\nUnsupported / recorded-only:')
fd = obs['first_drip_s']
print('  first_drip_s            : UNAVAILABLE —', fd['reason'])
print('  first_modeled_solute_arrival_s:', obs['first_modeled_solute_arrival_s']['value'],
      's (numerical diagnostic, NOT physical first drip)')
print('  brew_temperature_c      :', recipe.brew_temperature_c, 'C (recorded-only)')

print('\nComponent coverage (executed vs eligible):')
for c in run.coverage:
    if c.executed:
        print(f'  EXECUTED    {c.component_id} ({c.stage})')
print('  ...plus', sum(1 for c in run.coverage if not c.executed),
      'calibration-only / separate-lens / excluded components (none executed or averaged).')

if run.warnings:
    print('\nDomain warnings (extrapolation):')
    for w in run.warnings:
        print('  -', w)

print('\nWhat this does NOT prove: not your actual puck, not a flavor/taste prediction, not the',
      "'best' recipe; wetting and physical first drip are not modeled; pressure is prescribed;",
      'temperature is recorded-only; alternative components are not blended into a consensus.')

## 9. Download the report files

Deterministic JSON, Markdown, figures, and the caption text equivalent. On Colab these download to your machine; elsewhere they remain in the `guided_pull_report/` directory.

In [ ]:
try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

for f in artifacts.files:
    print('wrote', f)
    if _in_colab:
        _colab_files.download(f)

In [ ]:
print('GUIDED_PULL_COMPLETE:', run.config.config_id, run.run_id, '·',
      obs['extraction_yield_pct']['value'], '% EY ·', len(run.traces), 'traces ·',
      len(artifacts.files), 'files')